# Django Database Connection

## Introduction to Custom Database Integration
---

In this lesson, you'll learn how to **connect Django to external databases** like MSSQL (Microsoft SQL Server).

This is essential when working with existing databases or when you need more power than SQLite provides.

1. [Why Use External Databases](#Why-Use-External-Databases),
    - [SQLite vs Production Databases](#SQLite-vs-Production-Databases),
    - [When to Switch](#When-to-Switch),
2. [Setting Up MSSQL with Docker](#Setting-Up-MSSQL-with-Docker),
    - [Docker Compose Configuration](#Docker-Compose-Configuration),
    - [Starting the Database](#Starting-the-Database),
3. [Django Database Configuration](#Django-Database-Configuration),
    - [DATABASES Settings](#DATABASES-Settings),
    - [Installing Database Drivers](#Installing-Database-Drivers),
4. [Environment Variables for Credentials](#Environment-Variables-for-Credentials),
    - [Why Environment Variables](#Why-Environment-Variables),
    - [Using python-decouple](#Using-python-decouple),
5. [Testing the Connection](#Testing-the-Connection),
    - [Verifying Connection](#Verifying-Connection),
    - [Running Migrations](#Running-Migrations),
6. [Migrating Data from SQLite to MSSQL](#Migrating-Data-from-SQLite-to-MSSQL),
    - [Step 1 — Backup SQLite data](#Step-1-—-Backup-SQLite-data),
    - [Step 2 — Switch to MSSQL](#Step-2-—-Switch-to-MSSQL),
    - [Step 3 — Load data into MSSQL](#Step-3-—-Load-data-into-MSSQL),
7. [Backing Up MSSQL Data](#Backing-Up-MSSQL-Data),
    - [Django dumpdata](#Django-dumpdata),
    - [Native MSSQL Backup](#Native-MSSQL-Backup),
8. [🧠 EXERCISE 🧠, Connect Your Blog to MSSQL](#🧠-EXERCISE-🧠,-Connect-Your-Blog-to-MSSQL).

<br>

## Why Use External Databases

---

### SQLite vs Production Databases

---

Django uses **SQLite** by default - it's simple, requires no setup, and stores everything in a single file.

<br>

However, SQLite has limitations:

| Feature | SQLite | MSSQL |
|---------|--------|-------|
| Concurrent writes | Limited | Excellent |
| Data types | Basic | Rich (JSON, XML, etc.) |
| Full-text search | Basic | Advanced |
| Scalability | Single file | Distributed possible |
| Production use | Development only | Production ready |

<br>

### When to Switch

---

**Switch to MSSQL when:**

- You're deploying to production
- You need concurrent access (multiple users writing)
- You need advanced features (JSON fields, full-text search)
- You're integrating with an existing database
- Your team uses a shared database

<br>

**Keep SQLite for:**

- Local development (simple setup)
- Testing (fast, isolated)
- Small single-user applications
- Prototyping

<br>

## Setting Up MSSQL with Docker

---

The easiest way to run MSSQL locally is with **Docker**. No installation, no configuration conflicts.

<br>

Database Portability with Docker: We use Docker Compose to create a consistent MSSQL environment regardless of the host operating system. This enables us to practice migrating from a development database (SQLite) to a professional production database (MSSQL).

<br>

### Docker Compose Configuration

---

Create a `docker-compose.yml` file in your project root:

```yaml
# docker-compose.yml

services:
  db:
    image: mcr.microsoft.com/mssql/server:2022-latest
    container_name: blog_db
    environment:
      ACCEPT_EULA: ${ACCEPT_EULA}
      SA_PASSWORD: ${SA_PASSWORD}
      MSSQL_PID: ${MSSQL_PID}
    ports:
      - "1433:1433"
    volumes:
      - mssql_data:/var/opt/mssql

volumes:
  mssql_data:
```

**Environment Variables (.env):** Always store sensitive information (like `SECRET_KEY`, API credentials, or database passwords) in a local `.env` file.

**Key Benefits:**
- **Security:** Prevents sensitive data from being leaked on GitHub/GitLab.

- **Flexibility:** Allows you to use different configurations for development, testing, and production environments without modifying the source code."

<br>

**Key settings explained:**

| Setting | Description |
|---------|-------------|
| `image: mcr.microsoft.com/mssql/server:2022-latest` | MSSQL Server 2022 (official Microsoft image) |
| `ACCEPT_EULA: "Y"` | Required — accepts Microsoft's license agreement |
| `SA_PASSWORD` | Password for the `SA` (System Administrator) account |
| `MSSQL_PID: "Developer"` | Free Developer edition (full features, not for production) |
| `ports: "1433:1433"` | Map container port to host |
| `volumes` | Persist data between restarts |

> **Note:** `SA_PASSWORD` must meet MSSQL complexity requirements: min. 8 characters, uppercase, lowercase, digit, special character.

<br>

### Starting the Database

---

**Start MSSQL:**

```bash
# Start in background
docker compose up -d

# Check if running
docker compose ps

# View logs
docker compose logs db
```

<br>

**Stop MSSQL:**

```bash
# Stop containers
docker compose down

# Stop and remove data (fresh start)
docker compose down -v
```

<br>

**Create the database:**

Docker Compose starts the MSSQL server, but it does **not** create the `blog_db` database automatically. You need to create it manually:

```bash
# Create the blog_db database
docker compose exec db /opt/mssql-tools18/bin/sqlcmd -S localhost -U SA -P "Blog_pass1" -No -Q "CREATE DATABASE blog_db;"
```

> **Note:** The `container_name: blog_db` in `docker-compose.yml` is just the name of the Docker container, not the database. The database must be created separately inside the running MSSQL server.

<br>

**Connect to MSSQL directly (for debugging):**

```bash
# Connect using sqlcmd inside container
docker compose exec db /opt/mssql-tools18/bin/sqlcmd -S localhost -U SA -P "Blog_pass1" -No

# Inside sqlcmd:
SELECT name FROM sys.databases;   -- List databases
GO
USE blog_db;                       -- Switch to database
GO
SELECT * FROM sys.tables;         -- List tables
GO
EXIT
```

<br>

## Django Database Configuration

---

### DATABASES Settings

---

Django's database configuration lives in `settings.py` under the `DATABASES` dictionary.

<br>

**Default SQLite configuration:**

In [ ]:
# mysite/mysite/settings.py (default)

DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.sqlite3',
        'NAME': BASE_DIR / 'db.sqlite3',
    }
}

<br>

**MSSQL configuration:**

In [ ]:
# mysite/mysite/settings.py (MSSQL) - without .env
# DON'T do this! Credentials should be in .env file, not in settings.py

DATABASES = {
    'default': {
        'ENGINE': 'mssql',
        'NAME': 'blog_db',
        'USER': 'SA',
        'PASSWORD': 'Blog_pass1',       # <-- NEVER hardcode passwords!
        'HOST': 'localhost',
        'PORT': '1433',
        'OPTIONS': {
            'driver': 'ODBC Driver 18 for SQL Server',
            'extra_params': 'TrustServerCertificate=yes',
        },
    }
}

# Instead, add these to your .env file:
#
#   DB_ENGINE=mssql
#   DB_NAME=blog_db
#   DB_USER=SA
#   DB_PASSWORD=Blog_pass1
#   DB_HOST=localhost
#   DB_PORT=1433
#   DB_DRIVER=ODBC Driver 18 for SQL Server
#
# And use python-decouple in settings.py (see cell below)

In [ ]:
import os

DATABASES = {
    'default': {
        'ENGINE': os.environ.get('DB_ENGINE', 'django.db.backends.sqlite3'),
        'NAME': os.environ.get('DB_NAME', BASE_DIR / 'db.sqlite3'),
        'USER': os.environ.get('DB_USER', ''),
        'PASSWORD': os.environ.get('DB_PASSWORD', ''),
        'HOST': os.environ.get('DB_HOST', ''),
        'PORT': os.environ.get('DB_PORT', ''),
    }
}

if os.environ.get('DB_DRIVER'):
    DATABASES['default']['OPTIONS'] = {
        'driver': os.environ.get('DB_DRIVER'),
        'extra_params': 'TrustServerCertificate=yes',
    }

<br>

**Configuration parameters:**

| Parameter | Description | Example |
|-----------|-------------|---------|
| `ENGINE` | Database backend | `mssql` |
| `NAME` | Database name | `blog_db` |
| `USER` | Database user | `SA` |
| `PASSWORD` | User password | `Blog_pass1` |
| `HOST` | Database host | `localhost` or IP |
| `PORT` | Database port | `1433` (MSSQL default) |
| `OPTIONS.driver` | ODBC driver name | `ODBC Driver 18 for SQL Server` |
| `OPTIONS.extra_params` | Additional connection params | `TrustServerCertificate=yes` |

<br>

## Installing Database Drivers

---

Django needs a **database driver** to communicate with the database.

<br>

Database Drivers: Software that acts as an intermediary or 'translator' between Django and your specific database engine (MSSQL).

Why it’s necessary: Django’s ORM is database-agnostic, meaning it needs a driver to translate generic Python queries into the specific SQL syntax required by Microsoft SQL Server.

<br>

**For MSSQL — two steps:**

**1. Install ODBC Driver for SQL Server** (system-level, outside Python):

- **macOS:**
```bash
brew install microsoft/mssql-release/msodbcsql18
```

- **Ubuntu/Debian:**
```bash
curl https://packages.microsoft.com/keys/microsoft.asc | sudo apt-key add -
curl https://packages.microsoft.com/config/ubuntu/22.04/prod.list | sudo tee /etc/apt/sources.list.d/mssql-release.list
sudo apt-get update
sudo ACCEPT_EULA=Y apt-get install -y msodbcsql18
```

<br>

**2. Install Python packages:**

```bash
uv add mssql-django pyodbc
```

> `mssql-django` is the official Microsoft Django backend for SQL Server. `pyodbc` is the underlying ODBC connector.

<br>

**Update requirements.txt:**

```bash
pip freeze > requirements.txt
```

<br>

### Using python-decouple

---

`python-decouple` is a popular library for managing configuration.

<br>

**Install:**

```bash
uv add python-decouple

# pip install ..
```

<br>

**Update `settings.py`:**

In [ ]:
# mysite/mysite/settings.py

import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

BASE_DIR = Path(__file__).resolve().parent.parent

# Security settings
SECRET_KEY = os.environ.get('SECRET_KEY', default='abc')  # TODO
DEBUG = os.environ.get('DEBUG', default='False')

# Database configuration
DATABASES = {
    'default': {
        'ENGINE': os.environ.get('DB_ENGINE', 'django.db.backends.sqlite3'),
        'NAME': os.environ.get('DB_NAME', BASE_DIR / 'db.sqlite3'),
        'USER': os.environ.get('DB_USER', ''),
        'PASSWORD': os.environ.get('DB_PASSWORD', ''),
        'HOST': os.environ.get('DB_HOST', ''),
        'PORT': os.environ.get('DB_PORT', ''),
    }
}

# Add MSSQL-specific OPTIONS only when DB_DRIVER is set
# This allows switching between SQLite and MSSQL via .env
if os.environ.get('DB_DRIVER'):
    DATABASES['default']['OPTIONS'] = {
        'driver': os.environ.get('DB_DRIVER'),
        'extra_params': 'TrustServerCertificate=yes',
    }


**Note:** `load_dotenv()` reads the `.env` file and loads its values into `os.environ`.
Without it, `os.getenv()` only reads system environment variables — you would need to
export them manually in your shell:
```bash
export SECRET_KEY='my-secret-key'
export DB_ENGINE=mssql
export DB_NAME=blog_db
# ... etc.
```


<br>

**Add `.env` to `.gitignore`:**

```gitignore
# .gitignore

.env
*.env
.env.*
```

<br>

**Create `.env.example` for documentation:**

```ini
# .env.example (commit this file)

DEBUG=True
SECRET_KEY=generate-a-new-secret-key
DB_ENGINE=mssql
DB_NAME=blog_db
DB_USER=SA
DB_PASSWORD=your_strong_password
DB_HOST=localhost
DB_PORT=1433
DB_DRIVER=ODBC Driver 18 for SQL Server
```

<br>

## Migrating Data from SQLite to MSSQL

---

Before connecting Django to MSSQL, you need to **export your existing data** from SQLite first.

Django provides two commands that together handle data migration cleanly:

| Command | What it does |
|---------|-------------|
| `dumpdata` | Exports data from current DB to JSON |
| `loaddata` | Imports JSON data into current DB |

<br>

The full workflow:

```
SQLite (old)          JSON file            MSSQL (new)
─────────────         ─────────            ───────────
Blog, Comment   →   dumpdata   →   .json   →   loaddata   →   Blog, Comment
BlogReview                                                      BlogReview
```

<br>

> **Important:** You must dump data **while `.env` still points to SQLite** (DB variables commented out). Once you switch to MSSQL, Django can no longer read from SQLite.


<br>

### Step 1 — Backup SQLite data

---

Before switching to MSSQL, export your data while Django still uses **SQLite**.

<br>

**Make sure `.env` does NOT have DB variables active** (comment them out or remove them):

```ini
# .env — DB variables commented out, Django uses SQLite defaults

SECRET_KEY='your-secret-key'
ACCEPT_EULA=Y
SA_PASSWORD=Blog_pass1
MSSQL_PID=Developer

# DB_ENGINE=mssql
# DB_NAME=blog_db
# DB_USER=SA
# DB_PASSWORD=Blog_pass1
# DB_HOST=localhost
# DB_PORT=1433
# DB_DRIVER=ODBC Driver 18 for SQL Server
```

<br>

**Then export data:**

```bash
# Export users + blog app data (users needed for FK references)
python manage.py dumpdata auth.user blog --indent 2 > backup/full_data.json
```

<br>

> **Important:** Include `auth.user` in the dump! Models like `BlogReview` have a FK to `auth_user` — loading blog data without users will fail with `IntegrityError`.

<br>

> **Tip:** Always backup before switching databases. Keep the JSON file — it's your safety net.


<br>

### Step 2 — Switch to MSSQL

---

Uncomment the DB variables in `.env`:

```ini
# .env — now pointing to MSSQL

DB_ENGINE=mssql
DB_NAME=blog_db
DB_USER=SA
DB_PASSWORD=Blog_pass1
DB_HOST=localhost
DB_PORT=1433
DB_DRIVER=ODBC Driver 18 for SQL Server
```

<br>

Run migrations to create empty tables in MSSQL:

```bash
python manage.py migrate
```

<br>

Verify the connection before loading data:

```bash
python manage.py check --database default
```


<br>

### Step 3 — Load data into MSSQL

---

With MSSQL active, load the JSON backup:

```bash
python manage.py loaddata backup/full_data.json
# DELETE FROM blog_reviews;
# DELETE FROM blog_comment;
# DELETE FROM blog_blog;
# DELETE FROM auth_user;
```

<br>

**Verify data was loaded:**

```bash
python manage.py shell
```

```python
>>> from blog.models import Blog, Comment, BlogReview
>>> Blog.objects.count()
21
>>> Comment.objects.count()
3
>>> BlogReview.objects.count()
4
```

<br>

**Common `loaddata` errors:**

| Error | Cause | Solution |
|-------|-------|----------|
| `DeserializationError` | JSON references model that doesn't exist | Run `migrate` first |
| `IntegrityError` | FK points to user that doesn't exist | Include `auth.user` in dumpdata |
| `duplicate key` | Data already exists in target DB | Clear tables first or use `--ignorenonexistent` |

<br>

> **Note:** Always include referenced models in your dump. If `BlogReview` has a FK to `auth.user`, you must dump `auth.user blog` together, not just `blog` alone.


<br>

## Testing the Connection

---

### Verifying Connection

---

After configuring, verify Django can connect to the database.

<br>

**Method 1: Django check command**

```bash
python manage.py check --database default
```

Expected output:
```
System check identified no issues (0 silenced).
```

<br>

**Method 2: Django shell**

```bash
python manage.py shell
```

```python
>>> from django.db import connection
>>> connection.ensure_connection()
>>> print("Connected to:", connection.settings_dict['NAME'])
Connected to: blog_db
```

<br>

**Method 3: Show migrations (also verifies connection)**

```bash
python manage.py showmigrations
```

If connection fails, you'll see an error like:
```
django.db.utils.OperationalError: ('08001', '[08001] ...')
```

<br>

### Running Migrations

---

Once connected, run migrations to create tables:

```bash
# Create migration files (if not already done)
python manage.py makemigrations

# Apply migrations to database
python manage.py migrate
```

<br>

**Verify tables were created:**

```bash
# Connect to MSSQL
docker compose exec db /opt/mssql-tools18/bin/sqlcmd -S localhost -U SA -P "Blog_pass1" -No

# Switch to blog_db first!
USE blog_db;
GO

# List tables
SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_TYPE = 'BASE TABLE';
GO

# You should see Django's default tables plus your blog app tables:
# TABLE_NAME
# --------------------
# auth_group
# auth_user
# blog_blog
# blog_comment
# blog_reviews
# django_migrations
# ...
```

> **Note:** Without `USE blog_db; GO` you'll see system tables from the `master` database instead of your Django tables.

<br>

## Backing Up MSSQL Data

---

Your MSSQL database is now populated. Before any further changes — deploy, schema migration, refactor — always make a backup.

<br>

Two approaches, each for a different purpose:

| Approach | Tool | Restores to | Use when |
|----------|------|-------------|----------|
| **Django backup** | `dumpdata` | Any Django project with same models | Dev, testing, seeding |
| **Native backup** | `sqlcmd` + `.bak` | Same MSSQL instance | Production, full restore |

<br>

### Native MSSQL Backup

---

For production, use MSSQL's native backup format (`.bak`). This is a **full binary snapshot** of the database — faster, smaller, and handles everything including indexes, constraints, and stored procedures.

<br>

**Create a `.bak` backup via `sqlcmd`:**

```bash
# Run inside the Docker container
docker compose exec db /opt/mssql-tools18/bin/sqlcmd \
    -S localhost -U SA -P "Blog_pass1" -No \
    -Q "BACKUP DATABASE [blog_db] TO DISK = N'/var/opt/mssql/backup/blog_db.bak' WITH FORMAT, INIT, NAME = 'blog_db full backup';"
```

<br>

**Copy the backup file out of the container:**

```bash
# Copy from container to local machine
docker cp blog_db:/var/opt/mssql/backup/blog_db.bak ./backup/blog_db.bak
```

<br>

**Restore from `.bak`:**

```bash
# Restore the database from backup
docker compose exec db /opt/mssql-tools18/bin/sqlcmd \
    -S localhost -U SA -P "Blog_pass1" -No \
    -Q "RESTORE DATABASE [blog_db] FROM DISK = N'/var/opt/mssql/backup/blog_db.bak' WITH REPLACE;"
```

<br>

**Comparison: Django dumpdata vs Native backup:**

| | Django `dumpdata` | Native `.bak` |
|---|---|---|
| Format | JSON (text) | Binary |
| Speed | Slower (Python) | Fast (native MSSQL) |
| Portability | Any Django + same models | Same MSSQL version |
| Includes | Data only | Data + schema + indexes |
| Restore target | Any DB engine | MSSQL only |
| Best for | Dev/staging seeding | Production backup |

<br>

> **Production tip:** Schedule native backups with a cron job or use Azure Backup when running MSSQL in the cloud. Never rely solely on `dumpdata` for production disaster recovery.

<br>

### Django dumpdata

---

`dumpdata` exports your data as JSON — portable, readable, and loadable into any Django project with the same models.

<br>

**Export the blog app:**

```bash
# While connected to MSSQL
python manage.py dumpdata blog --indent 2 > backup/blog_mssql_backup.json
```

<br>

**Export everything (including users and permissions):**

```bash
python manage.py dumpdata --indent 2 --exclude contenttypes --exclude auth.permission \
    > backup/full_mssql_backup.json
```

> `--exclude contenttypes` and `--exclude auth.permission` prevent FK conflicts when restoring to a fresh database.

<br>

**Restore from backup:**

```bash
python manage.py loaddata backup/blog_mssql_backup.json
```

<br>

**When to use Django dumpdata:**

- Seeding a staging environment from production data
- Sharing test data with the team
- Moving data between environments (dev → staging)
- Quick snapshot before a risky migration

Which one to choose?

python manage.py dumpdata (The Portable Way): Serializes your data into JSON/XML.

Use when: Moving data between different database engines (e.g., SQLite → MSSQL) or creating small 'fixtures' for testing.

SQL BACKUP / Docker Snapshot (The Professional Way): Creates a binary copy of the database files.

Use when: Backing up production data, handling millions of rows, or moving data between two identical MSSQL instances (e.g., Dev Docker → Prod Server)."

<br>

## Common Connection Issues

---

| Error | Cause | Solution |
|-------|-------|----------|
| `Connection refused` | Database not running | `docker compose up -d` |
| `Login failed for user 'SA'` | Wrong password | Check `SA_PASSWORD` matches `.env` |
| `Cannot open database "X"` | Database not created | Create DB manually via `sqlcmd` |
| `No module named 'mssql'` | Driver not installed | `pip install mssql-django pyodbc` |
| `Data source name not found` | ODBC driver missing | Install `msodbcsql18` system package |
| `SSL Provider error` | Certificate issue | Add `TrustServerCertificate=yes` to `OPTIONS.extra_params` |

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **External DB** | MSSQL for production, SQLite for development |
| **Docker Compose** | Easy way to run MSSQL locally |
| **DATABASES setting** | ENGINE, NAME, USER, PASSWORD, HOST, PORT, OPTIONS |
| **Database drivers** | `mssql-django` + `pyodbc` + ODBC Driver 18 |
| **Environment variables** | Never hardcode credentials, use `.env` files |
| **python-dotenv** | `load_dotenv()` + `os.environ.get('DB_NAME')` to read from `.env` |
| **Conditional OPTIONS** | Only add MSSQL OPTIONS when `DB_DRIVER` is set |
| **Data migration** | `dumpdata` from SQLite first, then `loaddata` into MSSQL |
| **Testing connection** | `manage.py check --database default` |


<br>

### 🧠 EXERCISE 🧠, Connect Your Blog to MSSQL

---

Connect your existing `mysite` project to MSSQL:

1. Create `docker-compose.yml` with MSSQL configuration
2. Start the database with `docker compose up -d`
3. Install ODBC Driver 18 for SQL Server (system package)
4. Install `mssql-django` and `pyodbc`
5. Create `.env` file with database credentials (`DB_NAME=blog_db`)
6. Update `mysite/settings.py` to use environment variables
7. Add `.env` to `.gitignore`
8. Test connection with `python manage.py check --database default`
9. Run migrations and verify tables `blog_blog`, `blog_comment`, `blog_reviews` exist